# 365 Probabilidades - Dia #116
## Qual a probabilidade de alguém passar a vida inteira na mesma cidade?

**Tipo:** Demográfico
**Data de publicação:** 2026-10-07
**Ferramenta:** Python
**Decisão analisada:** Ficar ou mudar de cidade é a exceção ou a regra?
**Hashtag:** #365Probabilidades #Dia116

---

### 📖 A História

Nasci em Bauru. Com um ano fui para Santo Antônio do Monte, em Minas, e voltei ainda
criança. Saí de novo para a faculdade, em Presidente Prudente, e depois vieram
Piracicaba, São Paulo e Curitiba. Hoje estou outra vez em Bauru.

Moro na cidade em que nasci, e mesmo assim morei em seis. Fui ao Censo para saber
quantas pessoas fazem o caminho oposto e passam a vida inteira na mesma cidade.

Este dia faz parte da segunda fase do projeto. A pergunta nasce de uma história, e o
dado é procurado depois.

---

### 📚 O Conceito: o que o Censo pergunta

O questionário da amostra do Censo pergunta a cada morador se ele já residiu em outro
município. Quem responde que sim entra na contagem de "pessoas que já residiram fora do
município". Quem responde que não sempre morou onde está.

Essa definição tem uma consequência boa para a pergunta de hoje. Quem saiu e voltou
conta como quem já morou fora, e esse é o meu caso. O complemento mede exatamente quem
nunca saiu.

A "vida inteira" não pode ser observada em quem ainda está no meio dela. Por isso o
número do dia é lido nas pessoas de 70 anos ou mais, que são as que mais se aproximam
de uma vida completa.

---

### 🧮 O Modelo

Quatro camadas, todas sobre a contagem do Censo.

1. **O número do dia.** A proporção de pessoas de 70 anos ou mais que nunca moraram em
   outro município, em 2022 e em 2010.
2. **A curva por idade.** A proporção de quem já mudou de cidade em cada faixa etária,
   com uma função de saturação ajustada. A equação fica à vista.
3. **A sala.** Simulação de Monte Carlo de uma sala com 30 pessoas da mesma faixa de
   idade, para ver quantas nunca saíram da cidade.
4. **Mulheres e homens.** A mesma curva separada por sexo.

**Fontes:**
- IBGE, Censo Demográfico 2022, resultados da amostra. Tabela SIDRA 3182, "Pessoas que
  já residiram fora do município, por tempo ininterrupto de residência no município,
  sexo e grupos de idade". Períodos 2010 e 2022. Dados de 2022 marcados pelo IBGE como
  preliminares. Consulta em 03/10/2026.
- IBGE, Censo Demográfico 2022, universo. Tabela SIDRA 9514, "População residente, por
  sexo, idade e forma de declaração da idade". N = 203.080.756.
- IBGE, Censo Demográfico 2010. Tabela SIDRA 200, "População residente, por sexo,
  situação e grupos de idade". N = 190.755.799.

**Nota metodológica.** O fator ×0,80 do projeto não se aplica. Ele vale para proporções
de pesquisa de opinião autorrelatada, e aqui a fonte é o Censo.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from scipy.optimize import curve_fit

SEMENTE = 42
SORTEIOS = 200_000
SALA = 30

plt.rcParams['font.family'] = 'serif'
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.facecolor'] = 'white'

VERMELHO, DOURADO, VERDE, CINZA = '#c0392b', '#c8a84b', '#1a5f5a', '#6b6a64'

def mil(n):
    return f"{int(n):,}".replace(",", ".")

def vir(x, casas=1):
    return f"{x:.{casas}f}".replace(".", ",")

print("✅ Bibliotecas carregadas · semente", SEMENTE)

In [ ]:
# --- DADOS DO CENSO ---
# Numerador: SIDRA 3182, pessoas que já residiram fora do município (amostra).
#   https://apisidra.ibge.gov.br/values/t/3182/n1/all/v/1458/p/2022/c305/0/c2/all/c58/all
#   https://apisidra.ibge.gov.br/values/t/3182/n1/all/v/1458,1001458/p/2010,2022/c305/all/c2/all/c58/all
# Denominador 2022: SIDRA 9514, população residente por idade (universo).
# Denominador 2010: SIDRA 200, população residente por grupos de idade.
# A última faixa é "80 anos ou mais" (soma das faixas de 80 a 100+ no denominador).

FAIXAS = ['0 a 4', '5 a 9', '10 a 14', '15 a 19', '20 a 24', '25 a 29', '30 a 34',
          '35 a 39', '40 a 44', '45 a 49', '50 a 54', '55 a 59', '60 a 64', '65 a 69',
          '70 a 74', '75 a 79', '80 ou mais']
MEIO = np.array([2.5, 7.5, 12.5, 17.5, 22.5, 27.5, 32.5, 37.5, 42.5, 47.5, 52.5, 57.5,
                 62.5, 67.5, 72.5, 77.5, 86.0])   # ponto médio; 86 é escolha nossa para 80+

# 2022, total
FORA_2022 = np.array([968375, 2155059, 3633850, 4309917, 5675494, 6741709, 7417195,
                      8178079, 8452428, 7601066, 7235648, 6876144, 6106990, 5063793,
                      3862224, 2544771, 2972878])
POP_2022 = np.array([12704860, 13749440, 13674961, 14375942, 15466463, 15469723,
                     15473117, 16172791, 16072170, 13640112, 12598581, 11569106,
                     9944389, 7876232, 5858536, 3847379,
                     2475030 + 1329203 + 579729 + 165178 + 37814])

# 2022, mulheres
FORA_2022_M = np.array([474524, 1056683, 1788418, 2176262, 2937770, 3522741, 3876511,
                        4285322, 4436076, 4011025, 3828847, 3698038, 3314492, 2769611,
                        2143349, 1448819, 1826435])
POP_2022_M = np.array([6243171, 6738158, 6682215, 7058427, 7699157, 7842265, 7935832,
                       8345458, 8291111, 7091003, 6584190, 6149601, 5338555, 4288180,
                       3243186, 2189593, 1465178 + 835554 + 385388 + 114859 + 27244])
FORA_2022_H = FORA_2022 - FORA_2022_M
POP_2022_H = POP_2022 - POP_2022_M

# 2010, total
FORA_2010 = np.array([1955517, 2773079, 4029943, 4916043, 6276902, 7174649, 7380258,
                      7071634, 6935771, 6540766, 5896358, 5030413, 4031263, 3043599,
                      2352883, 1623036, 1808198])
POP_2010 = np.array([13806733, 14967767, 17167135, 16986788, 17240864, 17102917,
                     15744616, 13888191, 13008496, 11834647, 10134322, 8284433, 6503287,
                     4852789, 3744738, 2570686,
                     1661523 + 816803 + 319959 + 96430 + 22676])

TOTAL_FORA_2022, TOTAL_POP_2022 = 89795619, 203080756
TOTAL_FORA_2010, TOTAL_POP_2010 = 78840311, 190755799

# Tempo de residência de quem já morou fora (2022)
TEMPO = {'menos de 1 ano': 1729373, '1 ano': 5288424, '2 anos': 4198858,
         '3 anos': 3546084, '4 anos': 2964501, '5 anos': 3124975,
         '6 a 9 anos': 8226044, '10 anos ou mais': 60717360}

IDOSOS = slice(14, 17)        # 70 a 74, 75 a 79, 80 ou mais

print("=" * 70)
print("  DADOS DO CENSO · JÁ MOROU EM OUTRO MUNICÍPIO")
print("=" * 70)
print(f"\n  2022: {mil(TOTAL_FORA_2022)} de {mil(TOTAL_POP_2022)} "
      f"({vir(TOTAL_FORA_2022 / TOTAL_POP_2022 * 100)}%)")
print(f"  2010: {mil(TOTAL_FORA_2010)} de {mil(TOTAL_POP_2010)} "
      f"({vir(TOTAL_FORA_2010 / TOTAL_POP_2010 * 100)}%)")
print(f"\n  Conferência das somas por idade:")
print(f"    2022: numerador {mil(FORA_2022.sum())} · denominador {mil(POP_2022.sum())}")
print(f"    2010: numerador {mil(FORA_2010.sum())} · denominador {mil(POP_2010.sum())}")
print("    (diferença de 1 pessoa no numerador é arredondamento da expansão da amostra)")
print("=" * 70)

In [ ]:
# --- O MODELO ---

ja_2022 = FORA_2022 / POP_2022          # proporção que já morou em outro município
ja_2010 = FORA_2010 / POP_2010
nunca_2022 = 1 - ja_2022
nunca_2010 = 1 - ja_2010

# ── CAMADA 1 · o número do dia: 70 anos ou mais ──
def nunca_idosos(fora, pop):
    return 1 - fora[IDOSOS].sum() / pop[IDOSOS].sum()

p_dia = nunca_idosos(FORA_2022, POP_2022)
p_dia_2010 = nunca_idosos(FORA_2010, POP_2010)
p_geral = 1 - TOTAL_FORA_2022 / TOTAL_POP_2022

print("=" * 70)
print("  CAMADA 1 · A VIDA INTEIRA NA MESMA CIDADE (70 ANOS OU MAIS)")
print("=" * 70)
print(f"\n  2022: {mil(POP_2022[IDOSOS].sum())} pessoas de 70 anos ou mais")
print(f"        {mil(FORA_2022[IDOSOS].sum())} já moraram em outro município")
print(f"        nunca saíram: {vir(p_dia * 100)}%  (cerca de 1 em cada {vir(1 / p_dia)})")
for i in range(14, 17):
    print(f"          {FAIXAS[i]:<11} {vir(nunca_2022[i] * 100)}%")
print(f"\n  2010: nunca saíram {vir(p_dia_2010 * 100)}%")
print(f"  Diferença 2022 − 2010: {vir((p_dia - p_dia_2010) * 100)} pontos percentuais")
print(f"\n  Todas as idades, 2022: {vir(p_geral * 100)}% nunca moraram em outra cidade")
print("  (inclui crianças e jovens, que ainda não tiveram tempo de mudar)")

# ── CAMADA 2 · a curva por idade ──
def saturacao(idade, teto, ritmo):
    return teto * (1 - np.exp(-ritmo * idade))

def ajusta(p):
    (teto, ritmo), _ = curve_fit(saturacao, MEIO, p, p0=[0.7, 0.03])
    res = p - saturacao(MEIO, teto, ritmo)
    r2 = 1 - (res ** 2).sum() / ((p - p.mean()) ** 2).sum()
    rmse = np.sqrt((res ** 2).mean())
    return teto, ritmo, r2, rmse

teto22, ritmo22, r2_22, rmse22 = ajusta(ja_2022)
teto10, ritmo10, r2_10, rmse10 = ajusta(ja_2010)
idade_virada = -np.log(1 - 0.5 / teto22) / ritmo22

print("\n" + "=" * 70)
print("  CAMADA 2 · A CURVA POR IDADE")
print("=" * 70)
print("\n  Faixa         já morou fora (2022)   nunca saiu (2022)   já morou fora (2010)")
for i, f in enumerate(FAIXAS):
    print(f"  {f:<12}  {vir(ja_2022[i] * 100):>10}%   {vir(nunca_2022[i] * 100):>15}%   "
          f"{vir(ja_2010[i] * 100):>15}%")
print(f"\n  Função ajustada (2022): P(já mudou) = {vir(teto22, 3)} × (1 − e^(−{vir(ritmo22, 4)} × idade))")
print(f"    R² = {vir(r2_22, 3)} · erro típico = {vir(rmse22 * 100)} pontos percentuais")
print(f"    teto = {vir(teto22 * 100)}%  →  {vir((1 - teto22) * 100)}% nunca sairiam, pelo ajuste")
print(f"    idade em que a curva cruza 50%: {vir(idade_virada)} anos")
print(f"  Função ajustada (2010): teto = {vir(teto10 * 100)}% · ritmo = {vir(ritmo10, 4)} · "
      f"R² = {vir(r2_10, 3)} · erro típico = {vir(rmse10 * 100)} pp")

# ── CAMADA 3 · a sala de 30 pessoas ──
rng = np.random.default_rng(SEMENTE)
GRUPOS = {
    '20 a 24 anos': nunca_2022[4],
    '40 a 44 anos': nunca_2022[8],
    '70 anos ou mais': p_dia,
}
salas = {}
print("\n" + "=" * 70)
print(f"  CAMADA 3 · UMA SALA DE {SALA} PESSOAS DA MESMA IDADE ({mil(SORTEIOS)} sorteios)")
print("=" * 70)
for nome, p in GRUPOS.items():
    x = rng.binomial(SALA, p, size=SORTEIOS)
    lo, hi = np.percentile(x, [2.5, 97.5])
    maioria = (x > SALA / 2).mean()
    exata = 1 - stats.binom.cdf(SALA // 2, SALA, p)
    salas[nome] = dict(p=p, x=x, media=x.mean(), lo=lo, hi=hi, maioria=maioria, exata=exata)
    print(f"\n  {nome}: P(nunca saiu) = {vir(p * 100)}%")
    print(f"    nunca saíram, em média: {vir(x.mean())} de {SALA} · 95% das salas entre "
          f"{int(lo)} e {int(hi)}")
    print(f"    salas em que quem nunca saiu é maioria: {vir(maioria * 100)}% "
          f"(conta exata: {vir(exata * 100)}%)")

# ── CAMADA 4 · mulheres e homens ──
ja_m = FORA_2022_M / POP_2022_M
ja_h = FORA_2022_H / POP_2022_H
dif = (ja_m - ja_h) * 100
print("\n" + "=" * 70)
print("  CAMADA 4 · MULHERES E HOMENS (2022)")
print("=" * 70)
print(f"\n  Já moraram em outro município: mulheres {vir(FORA_2022_M.sum() / POP_2022_M.sum() * 100)}% · "
      f"homens {vir(FORA_2022_H.sum() / POP_2022_H.sum() * 100)}%")
print(f"  Maior diferença por faixa: {FAIXAS[int(dif.argmax())]} anos, "
      f"{vir(dif.max())} pontos a mais para as mulheres")
print(f"  70 anos ou mais, nunca saíram: mulheres "
      f"{vir(nunca_idosos(FORA_2022_M, POP_2022_M) * 100)}% · homens "
      f"{vir(nunca_idosos(FORA_2022_H, POP_2022_H) * 100)}%")

# ── Quem veio de fora ficou? ──
tot_tempo = sum(TEMPO.values())
print("\n" + "=" * 70)
print("  QUEM JÁ MOROU FORA · HÁ QUANTO TEMPO ESTÁ NA CIDADE ATUAL (2022)")
print("=" * 70)
print(f"\n  10 anos ou mais: {vir(TEMPO['10 anos ou mais'] / tot_tempo * 100)}%")
print(f"  5 anos ou menos: {vir(sum(v for k, v in TEMPO.items() if k not in ('6 a 9 anos', '10 anos ou mais')) / tot_tempo * 100)}%")
print("=" * 70)

In [ ]:
# --- VISUALIZAÇÃO ---
RODAPE = 'Fonte: IBGE, Censo Demográfico 2022 (SIDRA 3182 e 9514) e 2010 (SIDRA 200) | #365Probabilidades'
grade = np.linspace(0, 95, 400)

# ══ GRÁFICO 1 · a curva por idade ══
fig1, ax1 = plt.subplots(figsize=(12, 8))
ax1.plot(grade, saturacao(grade, teto22, ritmo22) * 100, color=VERMELHO, linewidth=2.5,
         label=f'ajuste: {vir(teto22, 2)} × (1 − e^(−{vir(ritmo22, 3)} × idade))')
ax1.scatter(MEIO, ja_2022 * 100, color=VERDE, s=70, zorder=3, label='Censo 2022, por faixa de idade')
ax1.axhline(50, color=CINZA, linestyle=':', linewidth=1.4)
ax1.text(1, 51.2, 'metade', fontsize=10, color=CINZA)
ax1.axhline(teto22 * 100, color=DOURADO, linestyle='--', linewidth=1.6)
ax1.text(1, teto22 * 100 + 1.2, f'teto do ajuste: {vir(teto22 * 100)}%', fontsize=10, color=DOURADO)
ax1.set_xlabel('Idade (ponto médio da faixa)', fontsize=12)
ax1.set_ylabel('Já morou em outro município (%)', fontsize=12)
ax1.set_ylim(0, 80)
ax1.set_xlim(0, 95)
ax1.set_title('Quanto mais idade, mais gente já mudou de cidade\n'
              'mas a curva para de subir perto de 65%', fontsize=14, pad=16)
ax1.legend(frameon=False, fontsize=11, loc='lower right')
plt.figtext(0.5, 0.01, RODAPE, ha='center', fontsize=9, color='gray')
plt.tight_layout(rect=(0, 0.03, 1, 1))
plt.savefig('dia-116-grafico-01-curva.png', dpi=150, bbox_inches='tight')
plt.close()
print("✅ Gráfico 1 salvo!")

# ══ GRÁFICO 2 · quem nunca saiu, por faixa ══
fig2, ax2 = plt.subplots(figsize=(12, 8))
cores = [VERMELHO if i >= 14 else CINZA for i in range(len(FAIXAS))]
barras = ax2.bar(FAIXAS, nunca_2022 * 100, color=cores, alpha=0.85)
for b, v in zip(barras, nunca_2022 * 100):
    ax2.text(b.get_x() + b.get_width() / 2, v + 1, f'{v:.0f}', ha='center', fontsize=9)
ax2.axhline(p_dia * 100, color=VERMELHO, linestyle='--', linewidth=1.4)
ax2.text(len(FAIXAS) - 0.4, p_dia * 100 + 6, f'70 anos ou mais\n{vir(p_dia * 100)}%', ha='right',
         fontsize=11, color=VERMELHO, fontweight='bold')
ax2.set_ylabel('Nunca morou em outro município (%)', fontsize=12)
ax2.set_xlabel('Faixa de idade (anos)', fontsize=12)
ax2.set_ylim(0, 100)
plt.setp(ax2.get_xticklabels(), rotation=45, ha='right')
ax2.set_title('Quem passou a vida na mesma cidade\n'
              'entre as pessoas de 70 anos ou mais, uma em cada três', fontsize=14, pad=16)
plt.figtext(0.5, 0.01, RODAPE, ha='center', fontsize=9, color='gray')
plt.tight_layout(rect=(0, 0.03, 1, 1))
plt.savefig('dia-116-grafico-02-nunca-saiu.png', dpi=150, bbox_inches='tight')
plt.close()
print("✅ Gráfico 2 salvo!")

# ══ GRÁFICO 3 · a sala de 30 pessoas ══
fig3, ax3 = plt.subplots(figsize=(12, 8))
cores_sala = {'20 a 24 anos': VERDE, '40 a 44 anos': DOURADO, '70 anos ou mais': VERMELHO}
for nome, s in salas.items():
    k = np.arange(0, SALA + 1)
    freq = np.bincount(s['x'], minlength=SALA + 1) / SORTEIOS * 100
    ax3.plot(k, freq, marker='o', markersize=4, linewidth=2, color=cores_sala[nome],
             label=f"{nome}: média {vir(s['media'])} de {SALA}")
    ax3.fill_between(k, freq, alpha=0.12, color=cores_sala[nome])
ax3.axvline(SALA / 2, color=CINZA, linestyle=':', linewidth=1.4)
ax3.text(SALA / 2 + 0.3, ax3.get_ylim()[1] * 0.93, 'metade da sala', fontsize=10, color=CINZA)
ax3.set_xlabel(f'Pessoas que nunca moraram em outra cidade, numa sala de {SALA}', fontsize=12)
ax3.set_ylabel('Salas simuladas (%)', fontsize=12)
ax3.set_title(f'Uma sala de {SALA} pessoas da mesma idade\n'
              f'{mil(SORTEIOS)} sorteios, semente {SEMENTE}', fontsize=14, pad=16)
ax3.legend(frameon=False, fontsize=11)
plt.figtext(0.5, 0.01, RODAPE, ha='center', fontsize=9, color='gray')
plt.tight_layout(rect=(0, 0.03, 1, 1))
plt.savefig('dia-116-grafico-03-sala.png', dpi=150, bbox_inches='tight')
plt.close()
print("✅ Gráfico 3 salvo!")

# ══ GRÁFICO 4 · 2010 contra 2022, e mulheres contra homens ══
fig4, (ax4, ax5) = plt.subplots(1, 2, figsize=(14, 7))
ax4.plot(MEIO, ja_2010 * 100, marker='s', color=CINZA, linewidth=2, label='Censo 2010')
ax4.plot(MEIO, ja_2022 * 100, marker='o', color=VERMELHO, linewidth=2, label='Censo 2022')
ax4.set_xlabel('Idade (ponto médio da faixa)', fontsize=11)
ax4.set_ylabel('Já morou em outro município (%)', fontsize=11)
ax4.set_ylim(0, 80)
ax4.set_title('2010 e 2022', fontsize=13)
ax4.legend(frameon=False, fontsize=10, loc='lower right')
ax5.plot(MEIO, ja_m * 100, marker='o', color=VERDE, linewidth=2, label='Mulheres')
ax5.plot(MEIO, ja_h * 100, marker='s', color=DOURADO, linewidth=2, label='Homens')
ax5.set_xlabel('Idade (ponto médio da faixa)', fontsize=11)
ax5.set_ylim(0, 80)
ax5.set_title('Mulheres e homens, 2022', fontsize=13)
ax5.legend(frameon=False, fontsize=10, loc='lower right')
fig4.suptitle('A mesma curva em dois censos e nos dois sexos', fontsize=14)
plt.figtext(0.5, 0.01, RODAPE, ha='center', fontsize=9, color='gray')
plt.tight_layout(rect=(0, 0.03, 1, 0.96))
plt.savefig('dia-116-grafico-04-comparacoes.png', dpi=150, bbox_inches='tight')
plt.close()
print("✅ Gráfico 4 salvo!")

### 💡 O Insight

**Entre os brasileiros de 70 anos ou mais, 34,4% nunca moraram em outra cidade.**

É cerca de uma pessoa em cada três, num grupo de 14,3 milhões. As três faixas de idade
que formam esse grupo dão quase o mesmo número, 34,1%, 33,9% e 35,2%.

A curva por idade mostra como se chega aí. Aos 20 e poucos anos, 63,3% ainda não
saíram. Entre 35 e 39 anos a conta passa da metade, com 50,6% já tendo morado em outro
município. Depois a subida perde força e fica perto de 65% a partir dos 70. A função
ajustada, 0,70 × (1 − e^(−0,034 × idade)), acompanha os dezessete pontos com erro típico
de 1,2 ponto percentual.

Na minha faixa, de 40 a 44 anos, 47,4% nunca saíram. Numa sala com 30 pessoas dessa
idade, a simulação dá em média 14 que sempre moraram na mesma cidade. Entre pessoas de
70 anos ou mais seriam 10 em 30, e em só 2,5% das salas elas formariam a maioria.

Dois detalhes do Censo completam o retrato. Mulheres mudam um pouco mais que homens,
45,5% contra 42,8%, com a maior distância entre 20 e 24 anos. Aos 70 a diferença
desaparece. E quem veio de fora tende a ficar, já que 67,6% dos que já moraram em outro
município estão há dez anos ou mais na cidade atual.

Em 2010 o número do dia era 37,4%. Em doze anos caiu três pontos.

Moro onde nasci e, para o Censo, faço parte dos 52,6% da minha idade que já moraram em
outra cidade. Ficar a vida inteira no mesmo lugar é menos comum do que mudar, e ainda
assim é a história de uma pessoa em cada três.

*Você ficou porque escolheu, ou porque nunca chegou a decidir?*

---

### ⚠️ Limitações do Modelo

- **A curva compara gerações, não acompanha pessoas.** Cada ponto é uma faixa de idade
  em 2022. Quem tem 70 anos hoje cresceu em outro país, com outra economia e outra
  migração. O número de 34,4% descreve essa geração e não é previsão para quem tem 40.
- **A vida inteira ainda não acabou.** As pessoas de 70 anos ou mais podem mudar de
  cidade depois do Censo, e o grupo só inclui quem estava vivo em 2022.
- **O Censo conta se a pessoa já morou fora, não quantas vezes.** Não existe no Censo o
  número de cidades em que alguém viveu. As minhas seis não têm com o que ser comparadas.
- **Município não é cidade em todos os casos.** Mudar de bairro dentro de um município
  grande não conta, e mudar para a cidade vizinha conta. Municípios criados por
  emancipação podem afetar a resposta, e não localizei como o IBGE trata esse caso.
- **Numerador e denominador vêm de fontes diferentes.** O numerador é da amostra do
  Censo 2022, que o IBGE marca como resultado preliminar. O denominador é do universo.
  As somas batem com o total do país. O erro amostral por faixa não foi lido e por isso
  o notebook não apresenta intervalo de confiança para as proporções.
- **Crianças em 2010 e em 2022.** Na faixa de 0 a 4 anos a proporção caiu de 14,2% para
  7,6% entre os dois censos, uma diferença grande que não sei explicar. A comparação
  entre censos foi usada só para o grupo de 70 anos ou mais.
- **A função ajustada é descritiva.** O teto de 70% é extrapolação do ajuste. Os dados
  observados param perto de 66%.
- **A sala simulada supõe pessoas independentes.** Numa sala de verdade as pessoas
  costumam ser da mesma cidade e da mesma história, e a variação seria maior.
- **O ponto médio de 86 anos para a faixa de 80 ou mais é escolha nossa.**

*A ciência é honesta sobre o que não sabe. O modelo também.*

---

### 📎 Links
- Substack: [link do post]
- Instagram: [link do post]

---
*365 Probabilidades · Decidindo com dados, um dia de cada vez.*